# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}\nPublished: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the Croissant `@id` (identifier) for all references.

In [ ]:
# List all record sets and corresponding fields by @id
record_set_ids = [r['@id'] for r in metadata.to_json().get('recordSet', [])]
if not record_set_ids:
    print("No record sets found in top-level 'recordSet'. Searching all top-level entries for record sets...")
    # Sometimes record sets reside inside distribution->hasPart, so check those
    record_set_ids = []
    for dist in metadata.to_json().get('distribution', []):
        if 'hasPart' in dist:
            for part in dist['hasPart']:
                if part.get('@type', '').lower() == 'recordset' or part.get('@type', '').endswith('RecordSet'):
                    record_set_ids.append(part['@id'])
if not record_set_ids:
    # Fallback: look for hasPart in metadata directly
    if 'hasPart' in metadata.to_json():
        for part in metadata.to_json()['hasPart']:
            if part.get('@type', '').lower() == 'recordset' or part.get('@type', '').endswith('RecordSet'):
                record_set_ids.append(part['@id'])
if not record_set_ids:
    print("No record sets found. The metadata may not have explicit record sets listed, or all data is under a single distribution.")
else:
    print("Available Record Sets by @id:")
    for rid in record_set_ids:
        print(f"  - {rid}")
    # For illustration, print field @ids for the first record set
    recset = record_set_ids[0]
    print(f"\nFields in Record Set '{recset}':")
    for record_set in metadata.to_json().get('recordSet', []):
        if record_set['@id'] == recset and 'field' in record_set:
            for field in record_set['field']:
                print(f"  - {field['@id']}")
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

If there are multiple record sets, load them all. If there are none explicitly listed, use the first available dataset.

In [ ]:
# In this dataset, record sets may not be explicitly listed in metadata.recordSet, but typically, you can retrieve available ones.
# We'll attempt to enumerate them; if not, we'll try to use the default (usually the main tabular data is the first set).
if not record_set_ids:
    # If there is only one (implicit) record set, it's usually the default
    print("No explicit record set @id found, using default/main record set.")
    record_sets = [None]
else:
    record_sets = record_set_ids

dataframes = {}
for record_set in record_sets:
    # When record_set is None, it loads the default/main table
    try:
        records = list(dataset.records(record_set=record_set))
        dataframes[record_set if record_set is not None else 'default'] = pd.DataFrame(records)
    except Exception as e:
        print(f"Could not load record set {record_set}: {e}")

# List columns in the (first/main) loaded data frame
main_key = record_sets[0] if record_sets[0] is not None else 'default'
print("Loaded columns:")
print(dataframes[main_key].columns.tolist())
dataframes[main_key].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

> **Note:** For demonstration, we'll use numeric fields such as 'Molecular.Weight' or 'MW' (commonly present in protein datasets), if they exist. Adjust field `@id`s as appropriate.

In [ ]:
# Select a numeric field for analysis
main_df = dataframes[main_key]

# Attempt to auto-select a likely numeric field
import numpy as np
candidate_numeric_fields = ['Molecular.Weight', 'MW', 'MolecularWeight', 'Abundance', 'Normalized.Abundance', 'Coverage(%)', 'Peptide.count', 'pI']
numeric_field = None
for field in candidate_numeric_fields:
    if field in main_df.columns:
        numeric_field = field
        break
if numeric_field is None:
    # Use first numeric column if none found
    numeric_columns = main_df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_columns:
        numeric_field = numeric_columns[0]
    else:
        raise ValueError("No numeric field found in the data for EDA.")

print(f"Using numeric field: {numeric_field}")
threshold = main_df[numeric_field].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field]) else 0
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.3f} (mean):")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field (e.g., 'Description', 'Sample', 'Protein.Group')
candidate_group_fields = ['Description', 'Sample', 'Experiment', 'Condition', 'Protein.Group', 'Accession']
group_field = None
for field in candidate_group_fields:
    if field in main_df.columns:
        group_field = field
        break

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (showing mean values):")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

> For demonstration, we'll show a histogram of the selected numeric field and, if possible, its relation to a group/category field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the numeric field distribution
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field].dropna(), bins=40, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If group_field exists and has reasonable unique count, plot boxplot
if group_field is not None and main_df[group_field].nunique() < 20:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=main_df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we have:
- Loaded and inspected the metadata and tabular records from the FAIR^2 mass spectrometry protein dataset
- Identified (explicit or implicit) record sets and fields using their Croissant `@id`s
- Extracted data and performed simple filtering, normalization, and grouping analyses
- Visualized distributions of numeric protein features

The structure and content of the dataset allow for diverse proteomics and systems biology analyses, such as comparisons of protein abundances, search for potential biomarkers, or evaluation of experimental reproducibility. For rigorous downstream analyses, please consult the referenced documentation and Croissant schema.